# testing rss.py

Run this against live DealNews feeds and see if the Rss_Service class actually works.

In [1]:
from pathlib import Path
import sys

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(root_dir / "src"))
root_dir

PosixPath('/Volumes/256 SSD/deal_hunter')

In [2]:
from deal_hunter.config import settings
print(f"Feed URLs: {settings.rss_feed_url}")

Feed URLs: ['https://www.dealnews.com/c142/Electronics/?rss=1', 'https://www.dealnews.com/c39/Computers/?rss=1', 'https://www.dealnews.com/f1912/Smart-Home/?rss=1']


## raw feedparser output first

feedparser returns entry dicts. the URL is not where you'd guess, so check the keys before writing code that assumes a field name.

In [3]:
import feedparser

feed = feedparser.parse(settings.rss_feed_url[0])
print(f"Feed title: {feed.feed.get('title', 'N/A')}")
print(f"Number of entries: {len(feed.entries)}")

Feed title: DealNews - Best Electronic Deals Online - Daily Electronics Deal
Number of entries: 26


In [4]:
# What keys does a single entry have?
entry = feed.entries[0]
print("Keys:", list(entry.keys()))

Keys: ['title', 'title_detail', 'links', 'link', 'summary', 'summary_detail', 'id', 'guidislink', 'published', 'published_parsed']


In [5]:
# Print the fields that matter
print("title")
print(entry.get("title"))
print()
print("link")
print(entry.get("link"))
print()
print("url")
print(entry.get("url"))  # feedparser doesn't store this as "url", so what does your code get?
print()
print("links([{}])")
for i, lnk in enumerate(entry.get("links", [])):
    print(f"  [{i}] {lnk}")
print()
print("summary(html)")
print(entry.get("summary", "")[:500])

title
Charging Devices at Woot: Up to 70% off + free shipping w/ Prime

link
https://www.dealnews.com/Charging-Devices-at-Woot-Up-to-70-off-free-shipping-w-Prime/21821821.html?iref=rss-c142

url
None

links([{}])
  [0] {'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.dealnews.com/Charging-Devices-at-Woot-Up-to-70-off-free-shipping-w-Prime/21821821.html?iref=rss-c142'}

summary(html)
<img src="https://d.dlnws.com/64599/1776165345-belkin-wireless-charging-station-u23d.jpeg?h=94&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="Get&#x20;deals&#x20;on&#x20;a&#x20;selection&#x20;of&#x20;charging&#x20;devices&#x20;in&#x20;this&#x20;sale&#x20;at&#x20;Woot.&#x20;We&#x27;ve&#x20;pictured&#x20;the&#x20;3-in-1&#x20;Charging&#x20;Station&#x20;with&#x20;Night&#x20;Light&#x20;for&#x20;&#x24;25.99&#x20;&#x28;low&#x20;by&#x20;&#


`entry.url` gives None. the real URL lives in `entry.link`, or `entry.links[0]["href"]` as a fallback.

## testing extract() on the raw summary HTML

In [6]:
from deal_hunter.services.rss import Rss_Service

rss = Rss_Service()

raw_summary = entry.get("summary", "")
print("before extract")
print(raw_summary[:300])
print()
print("after extract")
cleaned = rss.extract(raw_summary)
print(cleaned)
print(f"\nLength: {len(cleaned)}")

before extract
<img src="https://d.dlnws.com/64599/1776165345-belkin-wireless-charging-station-u23d.jpeg?h=94&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="Get&#x20;deals&#x20;on&#x20;a&#x20;selection&#x20;of&#x20;charging&#x20;devices&#x20;in&#x20;

after extract
Get deals on a selection of charging devices in this sale at Woot. We've pictured the 3-in-1 Charging Station with Night Light for $25.99 (low by $24). Sale ends April 29 at 1am ET. Shop Now at Woot! An Amazon Company

Length: 217


In [7]:
for i, e in enumerate(feed.entries[:5]):
    result = rss.extract(e.get("summary", ""))
    print(f"[{i}] {e['title'][:100]}")
    print(f"    after entry:{result[:100]}..." if len(result) > 100 else f">{result}")
    print()

[0] Charging Devices at Woot: Up to 70% off + free shipping w/ Prime
    after entry:Get deals on a selection of charging devices in this sale at Woot. We've pictured the 3-in-1 Chargin...

[1] SmallRig Foldable V Mount Battery Plate for Mirrorless/DSLR Cameras for $64 + free shipping
    after entry:That's a savings of $16. Buy Now at Amazon Features Tool-free quick installation Foldable 180° for s...

[2] Meifigno Magnetic Car Phone Holder for $9 + free shipping w/ Prime
    after entry:Apply coupon code "LEKJ8OXE" for a savings of $11. Buy Now at Amazon Features Vacuum suction for sta...

[3] Unlocked Google Pixel 10 128GB AI Android Smartphone for $549 + free shipping
    after entry:Get the Unlocked Google Pixel 10 128GB AI Android Smartphone at Amazon for $549. That's the lowest p...

[4] Vizio V4K70M-08 70” 4K UHD LED HDR Smart TV for $328 + free shipping
    after entry:Walmart offers the Vizio V4K70M-08 70” 4K UHD LED HDR Smart TV for $328. That's a savings of $200 an...



checking what div classes actually exist in the summary HTML.

In [8]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(entry["summary"], "html.parser")
for div in soup.find_all("div"):
    print(f"div classes: {div.get('class')}")

div classes: ['snippet', 'summary']


## scrape_entry() on one entry

In [9]:
deal = rss.scrape_entry(entry)
print(type(deal))
print(deal)
print()
print(deal.describe())
print(f"\nURL: {deal.url}")
print(f"Details length: {len(deal.details)}")
print(f"Features length: {len(deal.features)}")

<class 'deal_hunter.models.deals.ScrapedDeal'>
title='Charging Devices at Woot: Up to 70% off + free shipping w/ Prime' summary="Get deals on a selection of charging devices in this sale at Woot. We've pictured the 3-in-1 Charging Station with Night Light for $25.99 (low by $24). Sale ends April 29 at 1am ET. Shop Now at Woot! An Amazon Company" url='https://www.dealnews.com/Charging-Devices-at-Woot-Up-to-70-off-free-shipping-w-Prime/21821821.html?iref=rss-c142' details="more\n\n\nGet deals on a selection of charging devices in this sale at Woot. We've pictured the 3-in-1 Charging Station with Night Light for $25.99 (low by $24). Sale ends April 29 at 1am ET. \nShop Now at Woot! An Amazon Company" features=''

Title: Charging Devices at Woot: Up to 70% off + free shipping w/ Prime
Details: more


Get deals on a selection of charging devices in this sale at Woot. We've pictured the 3-in-1 Charging Station with Night Light for $25.99 (low by $24). Sale ends April 29 at 1am ET. 
Shop Now 

## scrape_feeds() on all three RSS feeds

In [10]:
deals = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    max_per_feed=3,
)
print(f"\nScraped {len(deals)} deals")


Scraped 9 deals


In [ ]:
for i, d in enumerate(deals[:3]):
    print(f"--- Deal {i} ---")
    print(d.describe())
    print()

In [ ]:
already_seen = {d.url for d in deals}
print(f"Known URLs: {len(already_seen)}")

deals_round2 = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    known_urls=already_seen,
    max_per_feed=3,
)
print(f"Round 2 scraped {len(deals_round2)} new deals")

Known URLs: 9
Round 2 scraped 1 new deals (should be 0 or very few)
